In [0]:
# PySpark script to generate synthetic debit card fraud detection dataset
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType
from pyspark.sql.functions import col, lit, concat, lpad
from datetime import datetime, timedelta
import random

# 1. Define the transaction schema
transaction_schema = StructType([
    StructField("transaction_id", StringType(), False),
    StructField("card_id", StringType(), False),
    StructField("timestamp", TimestampType(), False),
    StructField("amount", DoubleType(), False),
    StructField("latitude", DoubleType(), False),
    StructField("longitude", DoubleType(), False)
])

# 2. Generate 5,000 historical base transactions
# Simulate typical transactions within the same city (e.g., San Francisco area)
base_timestamp = datetime(2024, 1, 1, 0, 0, 0)
num_cards = 200
num_base_transactions = 5000

# Create base transaction data using local Python loop
base_data = []
for i in range(num_base_transactions):
    card_id = f"CARD_{str(i % num_cards).zfill(3)}"  # Distribute evenly across 200 cards
    transaction_id = f"TXN_{str(i).zfill(6)}"
    # Random timestamp within a 30-day window
    timestamp = base_timestamp + timedelta(minutes=random.randint(0, 30*24*60))
    # Typical transaction amount
    amount = round(random.uniform(5.0, 500.0), 2)
    # San Francisco area coordinates (with slight variation)
    latitude = 37.7749 + random.uniform(-0.1, 0.1)
    longitude = -122.4194 + random.uniform(-0.1, 0.1)
    
    base_data.append((transaction_id, card_id, timestamp, amount, latitude, longitude))

# Create DataFrame from base data
base_df = spark.createDataFrame(base_data, schema=transaction_schema)

# 3. Inject 5 explicit "velocity fraud" anomalies
# Each anomaly consists of 2 transactions: one in NYC, one in London, 5 minutes apart
fraud_data = []
fraud_card_ids = [f"CARD_{str(i).zfill(3)}" for i in range(5)]  # Use cards 0-4 for fraud

for idx, card_id in enumerate(fraud_card_ids):
    # First transaction in New York
    fraud_timestamp_nyc = base_timestamp + timedelta(days=15+idx, hours=10)
    fraud_txn_id_nyc = f"FRAUD_NYC_{str(idx).zfill(2)}"
    fraud_data.append((
        fraud_txn_id_nyc,
        card_id,
        fraud_timestamp_nyc,
        round(random.uniform(100.0, 1000.0), 2),
        40.7128,  # New York latitude
        -74.0060  # New York longitude
    ))
    
    # Second transaction in London, 5 minutes later
    fraud_timestamp_london = fraud_timestamp_nyc + timedelta(minutes=5)
    fraud_txn_id_london = f"FRAUD_LON_{str(idx).zfill(2)}"
    fraud_data.append((
        fraud_txn_id_london,
        card_id,
        fraud_timestamp_london,
        round(random.uniform(100.0, 1000.0), 2),
        51.5074,  # London latitude
        -0.1278   # London longitude
    ))

# Create DataFrame from fraud data
fraud_df = spark.createDataFrame(fraud_data, schema=transaction_schema)

# 4. Combine base and fraud transactions
final_df = base_df.union(fraud_df).orderBy("timestamp")

# Register as temporary view
final_df.createOrReplaceTempView("tmp_transactions")

# Display top 10 rows
print(f"Total transactions generated: {final_df.count()}")
print(f"Base transactions: {num_base_transactions}")
print(f"Fraud anomalies: {len(fraud_data)} (5 velocity fraud patterns with 2 transactions each)")
print("\nTop 10 rows:")
final_df.show(10, truncate=False)

In [0]:
%sql
select count(*), card_id
from tmp_transactions
group by card_id
order by card_id